# How Do We Estimate Melt Density?
<br>
This lesson has been adapted from an excel-based version within the 
SSAC Physical Volcanology Modules collection, available through SERC (https://serc.carleton.edu/sp/ssac/volcanology/volcmodules.html). The collection was 
developed by Chuck Connor (University of South Florida) and Peter LaFemina (Penn State). Connor
is the original author of this particular lesson on melt density.

## Learning Goals
1. Use chemical analyses to understand melt properties and the nature of magma movement.
2. Use mole fraction, molar mass, and fractional volume of several different oxides to determine overall melt density.
3. Convert between molar and volumetric units.
4. Use partial molar volume, the coefficient of thermal expansion, and the coefficient of isothermal compressibility to calculate the molar volume 
   of each oxide.


## Background: Governing Equations

The density of a silicate melt can be estimated using the following equation:

$$\rho \;=\; \sum_{i=1}^{N}\frac{X_i\,M_i}{V_i}$$

| Symbol | Meaning | Unit |
|---|---|---|
| $\rho$ | melt density | g m⁻³ |
| $X_i$ | mole fraction of oxide $i$ | dimensionless |
| $M_i$ | molecular (molar) mass of oxide $i$ | g mol⁻¹ |
| $V_i$ | partial molar volume of oxide $i$ at $T,P$ | m³ mol⁻¹ |
| $N$   | total number of oxides in the melt |  |

So to calculate the density of a silicate melt, we need to know three things:

1) The mole fraction of each oxide in the melt.
2) The molecular mass of each oxide.
3) The fractional volume of each oxide under specific temperature and pressure conditions.

## Problem: 

> **What is the density of a basaltic silicate melt at $T = 1473$ K and $P = 1$ GPa?**

These conditions correspond to pressure and temperature conditions that might exist near the base of the crust, an area that must be traversed by a magma ascending from the mantle to the surface. In addition to a fixed temperature and pressure, you are given weight percentages of each oxide.

## Step 1: Create a Dataframe with the Given Data

First, save your tempurature and pressure values. Then, populate a dataframe with weight percent values and molar mass for each oxide. This is the data you will use for your calculation. Remember that the molar mass of each oxide corresponds to $M_i$ in the equation.

In [22]:
#Import necessary packages for data analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Variables for calculation
temperature_K = 1473.0   # temperature in kelvin
pressure_GPa  = 1.0      # pressure in gigapascals

#Given data (list of oxides, their weight percents, and their molar masses)
oxide_names = ['SiO2','TiO2','Al2O3','Fe2O3','FeO','MnO','MgO','CaO','Na2O','K2O','H2O','CO2'] #list of oxide names (column 1)
weight_percent = np.array([48.77, 1.15, 15.90, 1.33, 8.62, 0.17, 9.67, 11.16, 2.43, 0.08, 0.15, 0.10]) #list of weight percents, corresponding to the above oxides (column 2)
molar_mass_g_per_mol = np.array([60.08, 79.88, 101.96, 159.69, 71.85, 70.94, 40.30, 56.08, 61.99, 94.20, 18.02, 44.01]) #list of molar masses for each oxide (column 3)

#Create a dataframe of oxides and their attributes
melt_density_df = pd.DataFrame({ 
    'wt %': weight_percent,
    'molecular mass (g/mol)': molar_mass_g_per_mol,
}, index=pd.Index(oxide_names, name='Oxide'))
print(f'Temp (K) = {temperature_K:.0f}      Pressure (GPa) = {pressure_GPa:g}')
melt_density_df

Temp (K) = 1473      Pressure (GPa) = 1


,wt %,molecular mass (g/mol)
Oxide,,
SiO2,48.77,60.08
TiO2,1.15,79.88
Al2O3,15.90,101.96
Fe2O3,1.33,159.69
FeO,8.62,71.85
MnO,0.17,70.94
MgO,9.67,40.30
CaO,11.16,56.08
Na2O,2.43,61.99


## Step 2: Convert Weight Percent to Mole Fraction

Recall the $X_i$ component from the melt density equation -- mole fraction of an oxide. Currently, you have a weight percent and molar mass for each oxide.
In this step, you will use these values to calculate the mole fraction for each oxide.

In [23]:
# Divide an oxide's weight percent by its molar mass to obtain the # of moles of each oxide contained in 100 g of rock
moles_per_100g = weight_percent / molar_mass_g_per_mol

# Each oxide's share of the total moles (these add up to 1)
mole_fraction  = moles_per_100g / moles_per_100g.sum()

# Add columns for number of moles and mole fraction 
melt_density_df.insert(2,'mol', moles_per_100g)
melt_density_df.insert(3,'mole fraction', mole_fraction)

melt_density_df

,wt %,molecular mass (g/mol),mol,mole fraction
Oxide,,,,
SiO2,48.77,60.08,0.811751,0.506589
TiO2,1.15,79.88,0.014397,0.008984
Al2O3,15.90,101.96,0.155944,0.097320
Fe2O3,1.33,159.69,0.008329,0.005198
FeO,8.62,71.85,0.119972,0.074871
MnO,0.17,70.94,0.002396,0.001496
MgO,9.67,40.30,0.239950,0.149746
CaO,11.16,56.08,0.199001,0.124191
Na2O,2.43,61.99,0.039200,0.024463


## Step 3: Calculate Molar Volume Fraction

We now have almost all of the information we need to calculate the melt density. The final missing piece is the molar volume fraction ($V_i$) for each oxide.

For liquid silicate melts, the molar volume of each oxide is given by:

$$V_i(T,P, X) \;=\; \overline{V}_i \;+\; \left[\frac{\partial V_i}{\partial T}\right]_P (T-1673) \;+\; \left[\frac{\partial V_i}{\partial P}\right]_T P$$

| Symbol | Meaning | Unit |
|---|---|---|
| $\overline{V}_i \;$ | partial molar volume of oxide $i$ at 0.0001 GPa pressure and 1673 K temperature | m³ mol⁻¹|
| $\left[\frac{\partial V_i}{\partial T}\right]_P$ | coefficient of **thermal expansion** of oxide $i$ (how the molar volume changes with temperature at constant pressure) | m³ mol⁻¹ K⁻¹|
| $\left[\frac{\partial V_i}{\partial P}\right]_T$ | coefficient of **isothermal compressibility** of oxide $i$ (how the molar volume changes with pressure at constant temperature) | m³ mol⁻¹ GPa⁻¹|

Let's add this information to the dataframe. Below, you will see lists of given values for partial molar volume, thermal expansion, and compressibility. These have been estimated from experiments. Then, molar volume fraction will be calculated using these three sets of values.

In [24]:
# Define values for partial molar volume, thermal expansion, and compressibility in order of the oxides in our dataframe
reference_molar_volume  = np.array([26.75, 22.45, 37.80, 44.40, 13.94, 13.94, 12.32, 16.95, 29.03, 46.30, 26.27, 33.00])   # 1e-6 m3/mol
thermal_expansion_coeff = np.array([0.00,  7.24,  0.00,  9.09,  2.92,  2.92,  3.27,  3.74,  7.68, 12.08,  9.46,  0.00])     # 1e-9 m3/mol K
compressibility_coeff   = np.array([-1.89,-2.31, -2.26, -2.53,-0.45, -0.45,  0.27,  0.34, -2.40, -6.75, -3.15,  0.00])      # 1e-6 m3/mol GPa

# Calculate molar volume fraction according to the equation
molar_volume = ((reference_molar_volume  * 1e-6)
                + (thermal_expansion_coeff * 1e-9 * (temperature_K - 1673))
                + (compressibility_coeff   * 1e-6 * pressure_GPa))

# Add new columns to the dataframe
melt_density_df.insert(4,'Partial Mol Vol (1e-6 m3/mol)', reference_molar_volume)
melt_density_df.insert(5,'Thermal expansion (1e-9 m3/mol K)', thermal_expansion_coeff)
melt_density_df.insert(6,'Compressibility (1e-6 m3/mol GPa)', compressibility_coeff)
melt_density_df.insert(7,'Fractional Molar Volume (m3/mol)', molar_volume)

melt_density_df

,wt %,molecular mass (g/mol),mol,mole fraction,Partial Mol Vol (1e-6 m3/mol),Thermal expansion (1e-9 m3/mol K),Compressibility (1e-6 m3/mol GPa),Fractional Molar Volume (m3/mol)
Oxide,,,,,,,,
SiO2,48.77,60.08,0.811751,0.506589,26.75,0.00,-1.89,0.000025
TiO2,1.15,79.88,0.014397,0.008984,22.45,7.24,-2.31,0.000019
Al2O3,15.90,101.96,0.155944,0.097320,37.80,0.00,-2.26,0.000036
Fe2O3,1.33,159.69,0.008329,0.005198,44.40,9.09,-2.53,0.000040
FeO,8.62,71.85,0.119972,0.074871,13.94,2.92,-0.45,0.000013
MnO,0.17,70.94,0.002396,0.001496,13.94,2.92,-0.45,0.000013
MgO,9.67,40.30,0.239950,0.149746,12.32,3.27,0.27,0.000012
CaO,11.16,56.08,0.199001,0.124191,16.95,3.74,0.34,0.000017
Na2O,2.43,61.99,0.039200,0.024463,29.03,7.68,-2.40,0.000025


## Step 4: Calculate Melt Density

You now have all of the necessary information to implement the melt density equation. We want to report melt density in kg/m³, so pay close attention to the unit conversion below.

This cell forms $X_i M_i / V_i$ for each oxide, which is that oxide's contribution to the total
density in kg m⁻³, and sums the contributions. The molar mass is converted from g mol⁻¹ to
kg mol⁻¹ so the density comes out in SI units. The table assembles the full calculation in the
same column layout the tutorial spreadsheet uses, and the total melt density is printed in both
kg m⁻³ and g cm⁻³.

In [18]:
molar_mass_kg_per_mol = molar_mass_g_per_mol * 1e-3                             # convert g/mol to kg/mol
density_contribution  = mole_fraction * molar_mass_kg_per_mol / molar_volume    # each oxide's share of density, kg/m3
melt_density          = density_contribution.sum()                             # total melt density, kg/m3


melt_density_df.insert(8,'Fractional Density (kg/m3)', density_contribution)

print(f'melt density (kg/m3) = {melt_density:.2f}      ( {melt_density/1000:.4f} g/cm3 )')
melt_density_df

melt density (kg/m3) = 2982.34      ( 2.9823 g/cm3 )


,wt %,molecular mass (g/mol),mol,mole fraction,Partial Mol Vol (1e-6 m3/mol),Thermal expansion (1e-9 m3/mol K),Compressibility (1e-6 m3/mol GPa),Fractional Molar Volume (m3/mol),Fractional Density (kg/m3)
Oxide,,,,,,,,,
SiO2,48.77,60.08,0.811751,0.506589,26.75,0.00,-1.89,0.000025,1224.290898
TiO2,1.15,79.88,0.014397,0.008984,22.45,7.24,-2.31,0.000019,38.395034
Al2O3,15.90,101.96,0.155944,0.097320,37.80,0.00,-2.26,0.000036,279.198249
Fe2O3,1.33,159.69,0.008329,0.005198,44.40,9.09,-2.53,0.000040,20.723372
FeO,8.62,71.85,0.119972,0.074871,13.94,2.92,-0.45,0.000013,416.820044
MnO,0.17,70.94,0.002396,0.001496,13.94,2.92,-0.45,0.000013,8.220349
MgO,9.67,40.30,0.239950,0.149746,12.32,3.27,0.27,0.000012,505.592536
CaO,11.16,56.08,0.199001,0.124191,16.95,3.74,0.34,0.000017,421.026244
Na2O,2.43,61.99,0.039200,0.024463,29.03,7.68,-2.40,0.000025,60.432334


## Reflection Questions

This exercise estimated the density of a silicate melt at high temperature and pressure from its
composition and the thermodynamic properties of its oxides, conditions that are hard to reach in
the laboratory.

1. Work the units of $X_i M_i / V_i$ through by hand and confirm the result is a mass per unit
   volume (kg m⁻³).
2. How does this melt's density compare with typical crustal rock (about 2800 kg m⁻³), and what
   does that imply for magma buoyancy at the base of the crust?
3. Qualitatively, how would the density change if $T$ increased? If $P$ increased?


## References and Resources

Spera, F., 2000, Physical properties of magma, In: Sigurdsson et al., eds., Encyclopedia of Volcanoes, Academic Press, 171-190. (a very useful paper that discusses the physical properties of magma in detail)

For more advanced thermodynamics analysis and calculations, check out Thermobar (https://thermobar.readthedocs.io/en/latest/) and PetThermoTools (https://petthermotools.readthedocs.io/en/latest/). 

See a walkthrough of workflows available on VICTOR here: https://www.youtube.com/watch?v=nq5YTTcS9XI and here: https://www.youtube.com/watch?v=IITgAnbFmfk&list=PL8-R5SvyzkdVwoxOhZc4C4qF8LHlyoTg8&index=6